Core package import

In [1]:
import os
import chromadb
import streamlit as st
from dotenv import load_dotenv
from pathlib import Path

DISABLED - when running on streamlit cloud, there was a problem with cache so pointing it to custom /tmp

In [ ]:

##
#streamlit cloud running config to use tmp as cache
#os.environ['LLAMA_INDEX_CACHE_DIR'] = '/tmp/llama_index_cache'
#import nltk
#nltk_data_path = "/tmp/nltk_data"
#if not os.path.exists(nltk_data_path):
#    os.makedirs(nltk_data_path, exist_ok=True)
#nltk.data.path.insert(0, nltk_data_path)
#try:
#    nltk.data.find('tokenizers/punkt')
#except LookupError:
#    nltk.download('punkt', download_dir=nltk_data_path)
#    nltk.download('punkt_tab', download_dir=nltk_data_path)
##

RAG package import - llamaindex core, vector embeddings + store, llm, parser

In [ ]:

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from llama_index.core.node_parser import SentenceSplitter

Setting default page title for streamlit

In [2]:
st.set_page_config(page_title="Deutsche Telekom Press release RAG", layout="wide")

Loading .env file to read Groq API key

In [3]:
load_dotenv()
api_key = os.getenv("API_KEY")

Adding a sidebar with RAG settings - chosing from one of 3 LLM models provided for text search and one embedding model as default

In [4]:
#llm settings
st.sidebar.title("Settings")
llm_model_options = {
    "Llama 3.3 70B": "llama-3.3-70b-versatile",
    "GPT-OSS 120B": "openai/gpt-oss-120b",
    "Kimi K2 Instruct": "moonshotai/kimi-k2-instruct-0905"
}
emb_model_options = {
    "bge-small-en-v1.5":"BAAI/bge-small-en-v1.5"
}

selected_llm_model = st.sidebar.selectbox(
    "LLM Model:",
    options=list(llm_model_options.keys()),
    index=0
)
selected_emb_model = st.sidebar.selectbox(
    "Emb Model:",
    options=list(emb_model_options.keys()),
    index=0
)
selected_llm_model_id = llm_model_options[selected_llm_model]
selected_emb_model_id = emb_model_options[selected_emb_model]

2026-02-26 21:44:43.394 
  command:

    streamlit run C:\Users\PHATT\PycharmProjects\DTSE-KE\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


LLM settings to point to one of the models selected above, using key from .env. Setting text splitter with chunk size and overlap

In [ ]:
Settings.llm = Groq(model=selected_llm_model_id, api_key=api_key)
Settings.embed_model = HuggingFaceEmbedding(model_name=selected_emb_model_id)
Settings.text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

Functions to handle /data files (if present) and to count them for reference (displayed in the sidebar)

In [ ]:
#db files check
def check_files():
    if not os.path.exists("./data") or not os.listdir("./data"):
        st.error("No files found in /data folder")
        st.stop()

def count_all_files():
    check_files()
    path = Path("./data")
    file_count = sum(1 for item in path.iterdir() if item.is_file())
    return file_count

Status section in the sidebar - active models and dataset. Button for clearing chat and restarting

In [ ]:
st.sidebar.divider()
st.sidebar.info(f"**Active Model:** {selected_llm_model}")
st.sidebar.info(f"**Embedding modeL:**  {selected_emb_model}")
st.sidebar.info(f"**Data:** {count_all_files()} files in ./data")

if st.sidebar.button("Restart"):
    st.session_state.messages = []
    st.rerun()

decorator to cache indexing objects to save time/effort when running the chat first time

In [ ]:
@st.cache_resource(show_spinner="Analyzing dataset")

RAG core - set db path, collection, vector and context storage. If collection is empty (running first time or after delete), checking if we have dataset in /data and then reading the folder and loading the files. Then chunking, converting to vectors and storing to db (chromadb folder). If collection is not empty, it skips and just initializes the db. Returns index

In [ ]:
def get_index():
    db = chromadb.PersistentClient(path="./chroma_db")
    chroma_collection = db.get_or_create_collection("my_documents")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    #if db exists
    if chroma_collection.count() == 0:
        check_files()

        documents = SimpleDirectoryReader("./data").load_data()
        index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
    else:
        index = VectorStoreIndex.from_vector_store(vector_store)

    return index

running indexing functions and starting chat engine with the settings for keeping context, using top 3 similar nodes (lower than usual because of the limit of free tier context window). Setting basic system prompt

In [ ]:
index = get_index()

#chat engine
chat_engine = index.as_chat_engine(
    chat_mode="context",
    similarity_top_k=3,
    system_prompt=(
        "You are a press assistant. Answer only on provided context. "
        "If you dont have the information, say sorry, i dont know, but ask me tomorrow"
    )
)

setting Title, message session state cache and chat history to be kept on the screen

In [ ]:
st.title("Deutsche Telekom Press release RAG")
if "messages" not in st.session_state:
    st.session_state.messages = []

#chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

User input condition if prompt not empty and saving into messages cache. Sending prompt to the engine. Dropdown expander shows top k results with their score. End appends the message into the message cache

In [ ]:
if prompt := st.chat_input("Ask something about recent press releases"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    #gen response
    with st.chat_message("assistant"):
        response = chat_engine.chat(prompt)
        st.markdown(response.response)

        #source
        with st.expander("Sources"):
            for i, node in enumerate(response.source_nodes):
                st.markdown(f"**Source {i + 1} (Match Score: {round(node.score, 3)}):**")
                st.caption(node.node.get_content()[:500] + "...")
                st.divider()

        st.session_state.messages.append({"role": "assistant", "content": response.response})